In [ ]:
%pip install \
  jupyter \
  notebook \
  ipykernel \
  gradio \
  pillow \
  numpy \
  requests

# Vision-Language Drone Client (Interactive)

This notebook implements a real-time client for a Webots drone controlled
by a Vision-Language Model (Qwen3-VL) running locally via LM Studio.

The system:
- Receives camera frames from Webots via TCP
- Sends images + a high-level mission prompt to a VLM
- Receives continuous motion intentions (movement, rotation)
- Sends low-level control commands back to the drone

The mission prompt can be modified interactively using a graphical interface.


In [2]:
import socket
import struct
import time
import base64
import io
import requests
import numpy as np
from PIL import Image
import re
import threading
import gradio as gr

# ================= CONFIG =================
WEBOTS_IP = "127.0.0.1"
WEBOTS_PORT = 9002

LMSTUDIO_URL = "http://localhost:1235/v1/chat/completions"
MODEL_ID = "qwen/qwen3-vl-8b"

FPS = 3
MAX_FORWARD = 0.5
MAX_YAW = 0.8
# =========================================


# ================= SOCKET =================
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.connect((WEBOTS_IP, WEBOTS_PORT))
print("[OK] Connected to Webots")


def recv_exact(n):
    data = b""
    while len(data) < n:
        packet = sock.recv(n - len(data))
        if not packet:
            raise ConnectionError("Socket closed")
        data += packet
    return data


# ================= STATE =================
running = False

current_mission = (
    "The drone should continuously fly following a figure-eight trajectory.\n"
    "A figure-eight consists of two connected smooth curves in opposite directions.\n\n"
    "Behavior rules:\n"
    "- Always maintain forward movement.\n"
    "- Introduce small rotation while moving forward to create curved paths.\n"
    "- Alternate the rotation direction over time to form two opposite loops.\n"
    "- Do not fly straight for long periods.\n"
)

BASE_PROMPT = (
    "You are controlling a drone. "
    "Decide how the drone should move using two values:\n"
    "- movement: a number between 0 and 1 indicating forward motion.\n"
    "- rotation: a number between -1 and 1 indicating rotation "
    "(negative = left, positive = right).\n\n"
    "Mission:\n"
)


# ================= CONTROL LOOP =================
def control_loop():
    global running, current_mission

    last_time = 0

    while True:
        now = time.time()
        if now - last_time < 1.0 / FPS:
            time.sleep(0.01)
            continue
        last_time = now

        # ---- recibir imagen ----
        header = recv_exact(8)
        w, h = struct.unpack("ii", header)

        img_bytes = recv_exact(w * h * 4)
        frame = np.frombuffer(img_bytes, np.uint8).reshape((h, w, 4))
        frame = frame[:, :, :3]

        image = Image.fromarray(frame)

        if not running:
            sock.send(b"0.0 0.0 0.0 0.0\n")
            continue

        # ---- preparar imagen para VLM ----
        buf = io.BytesIO()
        image.save(buf, format="PNG")
        img_b64 = base64.b64encode(buf.getvalue()).decode()

        payload = {
            "model": MODEL_ID,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{img_b64}"
                            }
                        },
                        {
                            "type": "text",
                            "text": BASE_PROMPT + current_mission +
                                    "\nRespond ONLY in this format:\n"
                                    "movement=<value>, rotation=<value>"
                        }
                    ]
                }
            ]
        }

        try:
            resp = requests.post(LMSTUDIO_URL, json=payload, timeout=60).json()
        except Exception as e:
            print("[ERROR] LM Studio:", e)
            continue

        answer = resp["choices"][0]["message"]["content"].lower()

        movement = 0.0
        rotation = 0.0

        m = re.search(r"movement\s*=\s*([0-9.]+)", answer)
        r = re.search(r"rotation\s*=\s*(-?[0-9.]+)", answer)

        if m:
            movement = float(m.group(1))
        if r:
            rotation = float(r.group(1))

        vx = movement * MAX_FORWARD
        yaw = rotation * MAX_YAW

        cmd = f"{vx} 0.0 0.0 {yaw}\n"
        sock.send(cmd.encode())

        print(f"[VLM] {answer} -> {cmd.strip()}")


threading.Thread(target=control_loop, daemon=True).start()


# ================= GRADIO =================
def start(mission):
    global running, current_mission
    current_mission = mission
    running = True
    return "Mission running"


def stop():
    global running
    running = False
    return "Mission stopped"


with gr.Blocks() as demo:
    gr.Markdown("## Drone Mission Control (Stable)")

    mission_box = gr.Textbox(
        label="Mission",
        value=current_mission,
        lines=10
    )

    with gr.Row():
        start_btn = gr.Button("START")
        stop_btn = gr.Button("STOP")

    status = gr.Textbox(label="Status")

    start_btn.click(start, mission_box, status)
    stop_btn.click(stop, None, status)

demo.launch()


[OK] Connected to Webots
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Exception in thread Thread-10 (control_loop):
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.0_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/threading.py", line 1081, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.0_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/threading.py", line 1023, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/xt/q9_j3z3d6nzc7_9dgzcgpmwr0000gn/T/ipykernel_87913/3010989946.py", line 89, in control_loop
    sock.send(b"0.0 0.0 0.0 0.0\n")
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe
